# 03 — Regime models (`EXP-H02`)

## 1. Objective

Test whether filtered two-state regime probabilities improve horizon-return density forecasts relative to the unconditional distribution retained by H1.

## 2. Hypothesis

Filtered probabilities \(\pi_{t\mid t}\) from a two-state Gaussian HMM improve the 5-, 15-, and 60-minute log-return density relative to that unconditional benchmark.

## 3. Null

The regime mixture adds no density-forecast value on that pre-registered family.

## 4. Information set

The filter sees only one-minute log returns known at or before the decision time. A return enters at the close that reveals it, so it is not part of the horizon label that starts at that close. Probabilities are the forward filter \(\pi_{t\mid t}\). Smoothed state probabilities are used only inside Baum-Welch on the training window and are not a forecast input. No same-bar high or low, and no local clock.

## 5. Dataset and version

`research/registry/datasets/btcusdt-1m.json`, loaded from `s3://trading-data-centheos` with AWS profile `trading`.

## 6. Data-quality checks

The dataset freeze counts duplicates, off-grid rows, bad prices, inconsistent OHLC, and missing minutes. A missing minute resets the Markov chain. Labels with no filtered probability are dropped and counted.

## 7. Label definition

`log(close[t + h]) - log(close[t])` only when both minute bars exist. A gap does not become a longer return. First-passage labels are not scored: the barrier definition is still an open question, so this experiment is density only.

## 8. Baseline

The nested baseline is the unconditional distribution recorded by `EXP-H01`. Student-t is that baseline when H1 retains it or its development validation selects it. Rejection of Student-t leaves Gaussian. An H1 record with no result scores Gaussian. This notebook does not treat the comparison as confirmation.

## 9. Model / method

Two-state Gaussian HMM on one-minute returns, ordered so state 0 has the lower variance. Horizon densities are a mixture of the same family as the H1 benchmark, weighted by the filtered probability at the decision time. A state without a usable effective sample fails that window rather than collapsing to one regime.

## 10. Training / validation / test split

Rolling 24-month train and 6-month validation windows, stepped by 6 months. The final 12 calendar months are sealed. A label whose outcome falls beyond a block boundary leaves that block.

## 11. Experiment configuration

`research.protocol.H2_CONFIG`. Its digest is printed below.


In [1]:
from pathlib import Path
import os
import sys

root = Path.cwd().resolve()
while not (root / "research" / "__init__.py").is_file() and root != root.parent:
    root = root.parent
if not (root / "research" / "__init__.py").is_file():
    raise RuntimeError("open this notebook from the trading repository")
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
os.environ["AWS_PROFILE"] = "trading"
os.environ["AWS_DEFAULT_REGION"] = "ap-southeast-2"
os.environ["DATA_STORE"] = "s3"
os.environ["S3_BUCKET"] = "trading-data-centheos"


In [2]:
from research.evaluate import align_to_frozen_dataset
from research.load import load_research_bars
from research.manifest import load_manifest
from research.protocol import H2_CONFIG

manifest_path = root / "research" / "registry" / "datasets" / "btcusdt-1m.json"
frozen = load_manifest(manifest_path)
bars, objects = load_research_bars(
    venue=H2_CONFIG["venue"],
    symbol=H2_CONFIG["symbol"],
    timeframe=H2_CONFIG["timeframe"],
    bucket=H2_CONFIG["bucket"],
    profile=H2_CONFIG["profile"],
    region=H2_CONFIG["region"],
)
if bars.empty:
    raise RuntimeError("BTCUSDT one-minute load returned no rows")
kept, report, appended = align_to_frozen_dataset(bars, frozen, objects)
print(
    f"frozen digest={frozen.digest} rows={report.rows_out} "
    f"collector_rows_after_freeze={appended}"
)


frozen digest=c1192eca8fa673c386b4ff3aad751f4f2d4895e2b02d7b693a42aac1ecd28188 rows=3573284


## 12–17. Evaluation

The next cell reports validation log score, PIT, and conditional VaR diagnostics for the regime mixture and the unresolved-or-retained unconditional benchmark. Robustness is the same comparison across rolling origins. Failed windows are counted. There is no economic claim. The only ablation is the nested unconditional density. First-passage is outside this experiment.


In [3]:
from research.evaluate import evaluate_h2, h2_log_score_comparison
from research.registry import config_digest, load_record

h1_record = load_record(root / "research" / "registry" / "EXP-H01.json")
print(f"H2 configuration {config_digest(H2_CONFIG)}")
evaluation = evaluate_h2(kept["close"], h1_record, H2_CONFIG)
comparison = h2_log_score_comparison(evaluation.scores)
print(
    f"benchmark={evaluation.benchmark} resolved={evaluation.benchmark_resolved} "
    f"gap_resets={evaluation.gap_resets} alignment_exclusions={evaluation.alignment_exclusions} "
    f"failed_or_skipped_windows={len(evaluation.failures)}"
)
if evaluation.failures:
    for failure in evaluation.failures:
        print(f"failure origin={failure['origin']} horizon={failure['horizon']} reason={failure['reason']}")
print("decision remains exploratory; this validation pass is not confirmation")
comparison


H2 benchmark is unresolved: H1 is open; exploratory H2 uses Gaussian until H1 retains or rejects


H2 configuration c4d73024ca14d5d15ec0ff25106a599b2fb15114357a391d34dbbf5dba3b5c4e


Dropped 1 labels with no filtered regime probability


RuntimeError: regime mixture produced a PIT outside [0, 1]

These figures draw the validation scores from the cell above. They do not refit the models and they do not read the sealed test.

1. Regime mixture minus the unconditional benchmark mean log score at each origin. A point above zero means the mixture scored that validation window more highly.
2. Share of validation returns below each model's own 5% VaR. For the mixture, that VaR depends on the filtered probability at the decision time. The horizontal line is the pre-registered 5% rate.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

scores = evaluation.scores.copy()
required = {"horizon_minutes", "origin", "model", "benchmark", "mean_log_score", "var_exceedance_rate"}
missing = required - set(scores.columns)
if missing:
    raise RuntimeError(f"H2 scores are missing columns: {sorted(missing)}")
benchmarks = scores["benchmark"].unique()
if len(benchmarks) != 1:
    raise RuntimeError(f"H2 plots need one benchmark, found {list(benchmarks)}")
benchmark = str(benchmarks[0])
if not {"regime_mixture", benchmark}.issubset(set(scores["model"])):
    raise RuntimeError(f"H2 plots need both models, found {sorted(scores['model'].unique())}")
scores["origin_date"] = pd.to_datetime(scores["origin"], format="%Y%m%d")
horizons = sorted(int(value) for value in scores["horizon_minutes"].unique())
horizon_color = {5: "#4c78a8", 15: "#54a24b", 60: "#e45756"}
model_color = {benchmark: "#4c78a8", "regime_mixture": "#f58518"}

wide = scores.pivot_table(
    index=["horizon_minutes", "origin", "origin_date"],
    columns="model",
    values="mean_log_score",
    aggfunc="first",
).reset_index()
wide["regime_minus_benchmark"] = wide["regime_mixture"] - wide[benchmark]

fig, ax = plt.subplots(figsize=(9, 4.5))
for horizon in horizons:
    part = wide.loc[wide["horizon_minutes"] == horizon].sort_values("origin_date")
    ax.plot(
        part["origin_date"],
        part["regime_minus_benchmark"],
        marker="o",
        color=horizon_color.get(horizon),
        label=f"{horizon} min",
    )
ax.axhline(0.0, color="0.35", linewidth=0.8)
ax.set_title(f"Regime mixture minus {benchmark} mean log score")
ax.set_xlabel("validation origin")
ax.set_ylabel("log score advantage")
ax.legend()
fig.tight_layout()
plt.show()

fig, axes = plt.subplots(1, len(horizons), figsize=(4.2 * len(horizons), 4), sharey=True)
for ax, horizon in zip(np.atleast_1d(axes), horizons):
    part = scores.loc[scores["horizon_minutes"] == horizon]
    ax.axhline(0.05, color="0.35", linewidth=0.8)
    for model in (benchmark, "regime_mixture"):
        series = part.loc[part["model"] == model].sort_values("origin_date")
        ax.plot(
            series["origin_date"],
            series["var_exceedance_rate"],
            marker="o",
            color=model_color[model],
            label=model,
        )
    ax.set_title(f"{horizon} min")
    ax.set_xlabel("validation origin")
axes = np.atleast_1d(axes)
axes[0].set_ylabel("5% VaR exceedance rate")
axes[0].legend()
fig.tight_layout()
plt.show()


## 18. Result

The displayed comparison is a development result. It is not a confirmatory result. It becomes eligible for confirmation only after H1 retains Gaussian or Student-t, or rejects Student-t and leaves Gaussian, and a later run explicitly reads the sealed test.

## 19. Decision

Exploratory. Do not mark `EXP-H02` retained from this notebook. Retention requires a resolved H1 benchmark, a new confirmatory run that reads the sealed test, and a record in the experiment registry.

## 20. Reusable implementation

`research.regimes` holds the filtered HMM, the regime mixture, and the H1 benchmark rule. `research.evaluate.evaluate_h2` runs the development comparison. This notebook only loads data, calls them, and renders the result.

## 21. Next experiment

Leave H2 proposed until the sealed comparison is run against the retained H1 benchmark. H3 can be designed in parallel. A first-passage version of H2 needs its own experiment id once a barrier definition is pre-registered.
